In [1]:
# conda env 'vis2'

import sys
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import glob
import at3d

In [2]:
def segment_cloud(cloud, cvm):
    cloud['volume'] = cloud.extinction > 0
    cloud['volume'].data = cloud['volume'].data.astype(np.int64)
    cloud['cth_cloud'] = cloud.volume & cvm.cth_cloud
    cloud['lcl_cloud'] = cloud.volume & cvm.lcl_cloud
    cloud['spc_cloud'] = cloud.volume & cvm.spc_cloud
    cloud['cth_clear'] = cloud.volume & cvm.cth_clear
    cloud['lcl_clear'] = cloud.volume & cvm.lcl_clear
    cloud['spc_clear'] = cloud.volume & cvm.spc_clear
    cloud['cloud'] = cloud.volume & cvm.cloud
    cloud['clear'] = cloud.volume & cvm.clear
    cloud['intersection'] = cloud.cth_cloud & cloud.lcl_cloud & cloud.spc_cloud & cloud.cth_clear & cloud.lcl_clear & cloud.spc_clear & cloud.cloud & cloud.clear
    cloud['union'] = cloud.cth_cloud | cloud.lcl_cloud | cloud.spc_cloud | cloud.cth_clear | cloud.lcl_clear | cloud.spc_clear | cloud.cloud | cloud.clear
    return cloud


In [3]:
CVM_file = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/CVM_default.nc'

cvm = xr.load_dataset(CVM_file)
cvm['lcl_mask'].data = cvm['lcl_mask'].data.astype(np.int64)
cvm['cloud_volume_mask'].data = cvm['cloud_volume_mask'].data.astype(np.int64)
cvm['cth_mask'].data[0,:,:] = cvm['cth_mask'].data[-1,:,:] = 0
cvm['cth_mask'].data[:,0,:] = cvm['cth_mask'].data[:,-1,:] = 0

cvm['cth_cloud'] = cvm.cth_mask & np.logical_not(cvm.lcl_mask) & np.logical_not(cvm.carved_mask)
cvm['lcl_cloud'] = np.logical_not(cvm.cth_mask) & cvm.lcl_mask & np.logical_not(cvm.carved_mask)
cvm['spc_cloud'] = np.logical_not(cvm.cth_mask) & np.logical_not(cvm.lcl_mask) & cvm.carved_mask
cvm['cth_clear'] = np.logical_not(cvm.cth_mask) & cvm.lcl_mask & cvm.carved_mask
cvm['lcl_clear'] = cvm.cth_mask & np.logical_not(cvm.lcl_mask) & cvm.carved_mask
cvm['spc_clear'] = cvm.cth_mask & cvm.lcl_mask & np.logical_not(cvm.carved_mask)
cvm['cloud'] = cvm.cth_mask & cvm.lcl_mask & cvm.carved_mask
cvm['clear'] = np.logical_not(cvm.cth_mask) & np.logical_not(cvm.lcl_mask) & np.logical_not(cvm.carved_mask)
cvm['clear'].data = cvm['clear'].data.astype(np.int64)
cvm['intersection'] = cvm.cth_cloud & cvm.lcl_cloud & cvm.spc_cloud & cvm.cth_clear & cvm.lcl_clear & cvm.spc_clear & cvm.cloud & cvm.clear
cvm['union'] = cvm.cth_cloud | cvm.lcl_cloud | cvm.spc_cloud | cvm.cth_clear | cvm.lcl_clear | cvm.spc_clear | cvm.cloud | cvm.clear


In [4]:
version_dir = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/AT3D_output/V00'

titles = [
    'default',
    'No CVM',
    'Dark Surf Unc',
    'LCL Only',
    'CTH Only',
    'SC Only'
]

iter_files = [
    '/default/iterations/iter*.nc',
    '/CVM/noCVM/iterations/iter*.nc',
    '/CVM/darksurf_noCVM/iterations/iter*.nc',
    '/CVM/LCL/iterations/iter*.nc',
    '/CVM/CTH/iterations/iter*.nc',
    '/CVM/SC/iterations/iter*.nc'
]

clouds = {}
for title, iter_file in zip(titles, iter_files):
    files = np.sort(glob.glob(version_dir + iter_file))
    sensors, solvers, rte_grid = at3d.util.load_forward_model(files[-1])
    cloud = solvers[0.67].medium['cloud']
    cloud = segment_cloud(cloud, cvm)
    clouds[title] = cloud

/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '61'. Max cell optical depth: '3.3198599815368652'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3

In [5]:
# sensors_def, solvers_def, rte_grid_def = at3d.util.load_forward_model(def_files[-1])
# sensors_unc, solvers_unc, rte_grid_unc = at3d.util.load_forward_model(unc_files[-1])

# cloud_def = solvers_def[0.67].medium['cloud']
# cloud_unc = solvers_unc[0.67].medium['cloud']

In [6]:
# cloud_def = segment_cloud(cloud_def, cvm)
# cloud_unc = segment_cloud(cloud_unc, cvm)

In [7]:
regions = ['cth_cloud', 'lcl_cloud', 'spc_cloud', 'cth_clear', 'lcl_clear', 'spc_clear', 'cloud', 'clear']
print('Cloud Volume By Region (pct of total)')
tmp = '{:<10} |'.format('Region')
for t in titles:
    tmp += ' {:<19} |'.format(t)
tmp += ' {:<20}'.format('CVM Volume')
print(tmp)
print('-'*(len(titles)*21 + 12 + 22))
for r in regions:
    tmp = '{:<10} |'.format(r)
    for t in titles:
        c = clouds[t]
        rvol = c[r].integrate(['x','y','z']).data
        tot_vol = c.volume.integrate(['x','y','z']).data
        tmp += ' {:<8.3f} ({:<7.2f}%) |'.format(rvol, 100*rvol/tot_vol)
    tmp += ' {:<20.3f}'.format(cvm[r].integrate(['x','y','z']).data)
    print(tmp)
print('-'*(len(titles)*21 + 12 + 22))
tmp = '{:<10} |'.format('TOTAL')
for t in titles:
    c = clouds[t]
    rvol = c[r].integrate(['x','y','z']).data
    tot_vol = c.volume.integrate(['x','y','z']).data
    tmp += ' {:<19.3f} |'.format(tot_vol)
tmp += ' {:<20.3f}'.format(cvm.union.integrate(['x','y','z']).data)
print(tmp)

Cloud Volume By Region (pct of total)
Region     | default             | No CVM              | Dark Surf Unc       | LCL Only            | CTH Only            | SC Only             | CVM Volume          
----------------------------------------------------------------------------------------------------------------------------------------------------------------
cth_cloud  | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.187    (0.12   %) | 0.000    (0.00   %) | 0.019    (0.04   %) | 0.000    (0.00   %) | 14.068              
lcl_cloud  | 0.000    (0.00   %) | 0.038    (0.04   %) | 1.845    (1.20   %) | 0.026    (0.05   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 23.562              
spc_cloud  | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 3.265               
cth_clear  | 0.000    (0.00   %) | 49.887   (57.49  %) | 87.692   (57.18  %) | 30.455   (60.95  %) | 0.000    (0.00   %) | 29.132   (50.9

In [8]:
# ## Relative Bias and Difference from WITHIN the CVM
# cond = cvm.cloud_volume_mask.data == 1
# ext_def = cloud_def.extinction.data[cond]
# ext_unc = cloud_unc.extinction.data[cond]
# rel_diff = 100 * (np.sum((ext_unc - ext_def)**2))**0.5 / (np.sum(ext_def)**2)**0.5
# rel_bias = 100 * (np.sum(np.abs(ext_unc)) - np.sum(np.abs(ext_def))) / (np.sum(np.abs(ext_def)))

# print('Relative Bias: {:.2f}'.format(rel_bias))
# print('Relative Diff: {:.2f}'.format(rel_diff))